# 04 · Pipeline ETL: de CSV a Parquet

**Objetivos**
- Definir un esquema explícito.
- Construir un flujo **Extract → Transform → Load**.
- Escribir datos en **Parquet** particionado.
- Comparar tamaño y velocidad de CSV vs Parquet y observar *partition pruning*.
- Ejecutar el mismo pipeline con `spark-submit`.

> Duración estimada: 30 min

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Fuera de Docker -> local[*]. Dentro del docker-compose -> spark://spark-master:7077
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

builder = (SparkSession.builder
           .appName("04_ETL")
           .master(MASTER)
           .config("spark.sql.shuffle.partitions", "8"))   # pocos datos -> pocas particiones
if os.environ.get("SPARK_DRIVER_HOST"):
    builder = builder.config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

DATA = os.path.abspath("../data")   # carpeta de datos del repositorio
print("Spark", spark.version, "| master:", sc.master)
print("Spark UI:", sc.uiWebUrl)

In [ ]:
import shutil, time
from pyspark.sql.types import *

SALIDA = os.path.abspath("../salida")      # carpeta ignorada por git
shutil.rmtree(SALIDA, ignore_errors=True)

## 1. Extract: esquema explícito

In [ ]:
esquema = StructType([
    StructField("id_venta",        IntegerType(), False),
    StructField("fecha",           DateType(),    True),
    StructField("id_cliente",      IntegerType(), True),
    StructField("id_producto",     IntegerType(), True),
    StructField("cantidad",        IntegerType(), True),
    StructField("precio_unitario", DoubleType(),  True),
    StructField("descuento",       DoubleType(),  True),
    StructField("canal",           StringType(),  True),
    StructField("metodo_pago",     StringType(),  True),
])

crudo = spark.read.csv(f"{DATA}/ventas.csv", header=True, schema=esquema)
productos = spark.read.json(f"{DATA}/productos.json")
clientes = spark.read.csv(f"{DATA}/clientes.csv", header=True, inferSchema=True)
crudo.printSchema()

## 2. Transform

In [ ]:
ventas = (crudo
    .filter(F.col("cantidad").isNotNull() & (F.col("cantidad") > 0))
    .fillna({"canal": "Desconocido"})
    .withColumn("total", F.round(F.col("cantidad") * F.col("precio_unitario") * (1 - F.col("descuento")), 2))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .join(productos.select("id_producto", "producto", "categoria"), "id_producto", "left")
    .join(F.broadcast(clientes.select("id_cliente", "ciudad", "region", "segmento")), "id_cliente", "left"))

print(f"Crudo: {crudo.count():,}  ->  Limpio: {ventas.count():,}")
ventas.show(3)

## 3. Load: Parquet particionado por año y mes

In [ ]:
(ventas.write
 .mode("overwrite")
 .partitionBy("anio", "mes")
 .parquet(f"{SALIDA}/ventas_parquet"))

for d in sorted(os.listdir(f"{SALIDA}/ventas_parquet/anio=2025"))[:4]:
    print(d, os.listdir(f"{SALIDA}/ventas_parquet/anio=2025/{d}")[:2])

Se guarda también una versión en CSV con fines comparativos:

In [ ]:
ventas.write.mode("overwrite").option("header", True).csv(f"{SALIDA}/ventas_csv")

def tamano_mb(ruta):
    total = 0
    for raiz, _, archivos in os.walk(ruta):
        total += sum(os.path.getsize(os.path.join(raiz, a)) for a in archivos if not a.endswith(".crc"))
    return total / 1024 / 1024

print(f"CSV    : {tamano_mb(f'{SALIDA}/ventas_csv'):.2f} MB")
print(f"Parquet: {tamano_mb(f'{SALIDA}/ventas_parquet'):.2f} MB")

## 4. Lectura: CSV vs Parquet
Parquet es un formato **columnar**: cuando una consulta requiere dos columnas, solo se leen esas dos. Además, conserva el esquema y estadísticas por bloque (The Apache Software Foundation, s.f.).

In [ ]:
def medir(fn, n=3):
    tiempos = []
    for _ in range(n):
        t = time.time(); fn(); tiempos.append(time.time() - t)
    return min(tiempos)

t_csv = medir(lambda: spark.read.csv(f"{SALIDA}/ventas_csv", header=True, inferSchema=True)
                       .groupBy("categoria").agg(F.sum("total")).collect())
t_pq  = medir(lambda: spark.read.parquet(f"{SALIDA}/ventas_parquet")
                       .groupBy("categoria").agg(F.sum("total")).collect())
print(f"CSV: {t_csv:.2f} s | Parquet: {t_pq:.2f} s")

> Con 20 000 filas la diferencia es reducida. Al generar un conjunto de datos mayor (`python scripts/generar_datos.py --ventas 2000000`) y repetir la medición, la diferencia aumenta de manera considerable.

## 5. Partition pruning
Al filtrar por la columna de partición, Spark **lee únicamente las carpetas necesarias**. En el plan de ejecución, esta optimización aparece como `PartitionFilters` (The Apache Software Foundation, 2026b).

In [ ]:
dic = spark.read.parquet(f"{SALIDA}/ventas_parquet").filter("mes = 12")
dic.explain()
print("Ventas en diciembre:", dic.count())

## 6. Reporte final

In [ ]:
reporte = (spark.read.parquet(f"{SALIDA}/ventas_parquet")
           .groupBy("region", "categoria")
           .agg(F.count("*").alias("ventas"), F.round(F.sum("total")).cast("long").alias("ingresos"))
           .orderBy(F.desc("ingresos")))
reporte.show(10)
reporte.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{SALIDA}/reporte")

## 7. Ejecutar el pipeline con `spark-submit`
El mismo flujo está empaquetado en `src/etl_ventas.py`. Desde la raíz del repositorio:

```bash
spark-submit --master local[4] src/etl_ventas.py --datos data --salida salida
```

Dentro de Docker:
```bash
docker compose exec jupyter spark-submit --master spark://spark-master:7077 \
    src/etl_ventas.py --datos /home/jovyan/work/data --salida /home/jovyan/work/salida
```

## Ejercicios propuestos

1. Escribir `ventas` en Parquet particionado por `region` y comprobar con `explain()` la poda de particiones al filtrar `region = 'Caribe'`.
2. Guardar el reporte en formato **JSON** y leerlo nuevamente.
3. Comparar `repartition(1)` y `coalesce(1)` antes de escribir, y describir la diferencia observada en la Spark UI (pestaña *Stages*).

In [ ]:
# Ejercicio 1

In [ ]:
spark.stop()

## Referencias

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html

The Apache Software Foundation. (s.f.). *Apache Parquet documentation* [Documentación técnica]. Recuperado el 8 de noviembre de 2026, de https://parquet.apache.org/docs/